In [2]:
from pathlib import Path
from dotenv import load_dotenv
from IPython.display import display, Markdown

In [3]:
from langgraph.graph import StateGraph, END
from langchain_openai import ChatOpenAI
from typing import TypedDict
import os

In [4]:
# Carregando as variáveis de ambiente
ENV_PATH = Path('../.env')
load_dotenv(dotenv_path=ENV_PATH)

True

In [14]:
class AgentState(TypedDict):
    messages: list
    result: str

llm = ChatOpenAI(model="gpt-4o", temperature=0)

def call_gpt(state: AgentState):
    messages = state["messages"]
    response = llm.invoke(messages)
    return {"result": response.content}

workflow = StateGraph(AgentState)

workflow.add_node("gpt", call_gpt)

workflow.set_entry_point("gpt")
workflow.add_edge("gpt", END)

app = workflow.compile()

result = app.invoke({
    "messages": [{"role": "user", "content": "Olá tudo bem?"}],
    "result": ""
})

print(result["result"])

Olá! Tudo bem, e com você? Como posso ajudar hoje?


In [10]:
mermaid_code = app.get_graph().draw_mermaid()
display(Markdown(f"```mermaid\n{mermaid_code}\n```"))

```mermaid
---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	gpt(gpt)
	__end__([<p>__end__</p>]):::last
	__start__ --> gpt;
	gpt --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc

```

In [5]:
print(app.get_graph().draw_ascii())

+-----------+  
| __start__ |  
+-----------+  
      *        
      *        
      *        
   +-----+     
   | gpt |     
   +-----+     
      *        
      *        
      *        
 +---------+   
 | __end__ |   
 +---------+   


## Busca Web

In [10]:
from langchain_community.tools import DuckDuckGoSearchRun

In [19]:
class AgentState(TypedDict):
    messages: list
    result: str
    search: str | None = None

llm = ChatOpenAI(model='gpt-4o', temperature=0)


def search_web(state: AgentState) -> AgentState:
    search = DuckDuckGoSearchRun(region='br-pt', max_results=5)
    last_msg = state['messages'][-1]['content']
    result = search.run(last_msg)
    state['search'] = result
    state['messages'].append({
        'role': 'system',
        'content': f'Resultado da pesquisa: {result}'
    })
    return state


def call_gpt(state: AgentState) -> AgentState:
    response = llm.invoke(state["messages"])
    state["result"] = response.content
    return state


workflow = StateGraph(AgentState)

workflow.add_node("search", search_web)
workflow.add_node("gpt", call_gpt)

workflow.set_entry_point("search")
workflow.add_edge("search", "gpt")
workflow.add_edge("gpt", END)

app = workflow.compile()

mermaid_code = app.get_graph().draw_mermaid()
display(Markdown(f"```mermaid\n{mermaid_code}\n```"))

```mermaid
---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	search(search)
	gpt(gpt)
	__end__([<p>__end__</p>]):::last
	__start__ --> search;
	search --> gpt;
	gpt --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc

```

In [20]:
print(app.get_graph().draw_ascii())

+-----------+  
| __start__ |  
+-----------+  
      *        
      *        
      *        
  +--------+   
  | search |   
  +--------+   
      *        
      *        
      *        
   +-----+     
   | gpt |     
   +-----+     
      *        
      *        
      *        
 +---------+   
 | __end__ |   
 +---------+   


In [21]:
prompt = "Quais são os dois times na final da Libertadores de 2025? E quando será o jogo?"

result = app.invoke({
    "messages": [{"role": "user", "content": prompt}],
    "result": "",
    "search": None
})

print("Resposta do modelo:\n", result["result"])
print("\nResultado da busca:\n", result["search"])

Resposta do modelo:
 A final da Copa Libertadores de 2025 será disputada entre os times brasileiros Palmeiras e Flamengo. O jogo está marcado para o dia 29 de novembro de 2025, no Estádio Monumental "U", em Lima, no Peru.

Resultado da busca:
 A final da Copa Libertadores de 2025 será a partida decisiva que decidirá o vencedor da Copa Libertadores de 2025 . Esta será a 66.ª final da Copa Libertadores , o campeonato sul-americano de futebol continental de clubes organizado pela Confederação Sul-Americana de Futebol (CONMEBOL). Está marcada para 29 de novembro e será disputada entre os brasileiros Sociedade Esportiva Palmeiras ... 5 days ago · Final define o primeiro tetracampeão brasileiro da Libertadores entre Palmeiras e Flamengo. Confira o retrospecto, os últimos jogos e os mercados de apostas da Betnacional. 1 day ago · Palmeiras e Flamengo disputam a grande Final da Libertadores 2025 neste sábado (29). O confronto ocorre às 18h ( de Brasília), no Estádio Monumental “U”, em Lima, no